# The Gender Conversion Gap: From STEM and Digital Participation to Labour-Market Power - EDA
notebook 01

Table of content:

- [Purpose and scope](#purpose-and-scope)
- [Sample structure](#sample-structure)
- [Missingness in the latest-wide dataset](#missingness-in-the-latest-wide-dataset)
- [Year diagnostics](#year-diagnostics)
- [Descriptive statistics](#descriptive-statistics)
- [Indicator distributions](#indicator-distributions)
- [Estonia benchmark](#estonia-benchmark)
- [Exploratory relationship plots](#exploratory-relationship-plots)
- [Indicator extremes: top and bottom countries](#indicator-extremes-top-and-bottom-countries)
- [EDA summary and implications for hypothesis testing](#eda-summary-and-implications-for-hypothesis-testing)

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

## <a id="purpose-and-scope"></a>Purpose and scope

This notebook explores the cleaned latest-available country-level dataset created in `00_feasibility_and_data_contract.ipynb`.

The goal of this EDA is to understand the structure, completeness, distributions, country scope, and Estonia-specific position in the data before formal hypothesis testing.

This notebook focuses on:

* sample structure: OECD members, non-member economies, and aggregate areas;
* missingness in the latest-wide dataset;
* year diagnostics for latest-available indicators;
* descriptive statistics for core variables;
* Estonia benchmarking against the OECD-member sample;
* distribution checks and possible outliers;
* exploratory visual inspection of hypothesis-related variable pairs.

Formal statistical hypothesis testing is not performed here. It will be conducted in `02_hypothesis_testing.ipynb`.

In [60]:
DATA_PATH = '/Users/avr/Jupiter/Gender Inequality Report - startisctic capstone/data/processed/'

df = pd.read_csv(DATA_PATH + "gender_conversion_gap_latest.csv")
data_dict = pd.read_csv(DATA_PATH + "data_dictionary.csv")
hypotheses = pd.read_csv(DATA_PATH + "hypothesis_mapping.csv")

# EU28 is an aggregate area, not an individual country/economy.
# Reclassify it before any sample structure summaries or working samples are created.
additional_aggregate_areas = ["EU28"]
df["is_aggregate_area"] = (
    df["is_aggregate_area"] |
    df["REF_AREA"].isin(additional_aggregate_areas)
)
df.loc[df["REF_AREA"].isin(additional_aggregate_areas), "sample_scope"] = "aggregate area"


In [3]:
print("Main dataset shape:", df.shape)
display(df.head())

print("\nData dictionary shape:", data_dict.shape)
display(data_dict.head())

print("\nHypothesis mapping shape:", hypotheses.shape)
display(hypotheses.head())

Main dataset shape: (60, 19)


,REF_AREA,is_oecd_member,is_aggregate_area,sample_scope,stem_women_share,coding_gap_youth_M_minus_F,labour_force_participation_gap_M_minus_F,gender_wage_gap_median,part_time_employment_gap_F_minus_M,women_senior_middle_mgmt_share,stem_women_share_year,coding_gap_youth_M_minus_F_year,labour_force_participation_gap_M_minus_F_year,gender_wage_gap_median_year,part_time_employment_gap_F_minus_M_year,women_senior_middle_mgmt_share_year,latest_year_min,latest_year_max,latest_year_span
0,ARG,False,False,non-member / partner / candidate,NaN,NaN,19.078,9.090909,26.545918,NaN,NaN,NaN,2023.0,2024.0,2019.0,NaN,2019.0,2024.0,5.0
1,AUS,True,False,OECD member,33.352506,NaN,7.650,10.671213,16.924777,39.39,2024.0,NaN,2025.0,2024.0,2025.0,2024.0,2024.0,2025.0,1.0
2,AUT,True,False,OECD member,29.190229,10.8682,8.144,11.062252,25.858781,33.79,2024.0,2025.0,2025.0,2024.0,2025.0,2023.0,2023.0,2025.0,2.0
3,BEL,True,False,OECD member,28.368823,9.9548,7.970,0.906909,18.302429,33.69,2024.0,2025.0,2025.0,2022.0,2025.0,2023.0,2022.0,2025.0,3.0
4,BGR,False,False,non-member / partner / candidate,36.155884,2.3700,9.686,2.360074,0.595541,NaN,2023.0,2025.0,2025.0,2024.0,2025.0,NaN,2023.0,2025.0,2.0



Data dictionary shape: (6, 16)


,layer,name,source,indicator_type,MEASURE,MEASURE_label,SDG_SERIES,SDG_SERIES_label,UNIT_MEASURE,unit_label,required_filters,calculation,final_unit,definition,direction,limitations
0,education_pipeline,stem_women_share,stem_graduates_women_share,direct_indicator,GRAD,Graduates,NaN,NaN,PT_ST_SUB,Percentage of students in the same subgroup,EDUCATION_LEV=ISCED11_5T8; EDUCATION_FIELD=F05...,Use OBS_VALUE directly,percent,Women as a percentage of tertiary STEM graduates.,Higher = more women represented among tertiary...,Captures graduation output in selected STEM fi...
1,digital_skills_pipeline,coding_gap_youth_M_minus_F,coding_gap_youth,computed_gender_gap,H1K_I,Individuals who have written computer code - l...,NaN,NaN,PT_POP,Percentage of population,"AGE=Y16T24; SEX in [M, F]",Male OBS_VALUE - female OBS_VALUE,percentage points,Gender gap in the share of 16-24-year-olds who...,Higher = larger male advantage in youth coding...,Measures coding activity among 16-24-year-olds...
2,work_access,labour_force_participation_gap_M_minus_F,labour_force_participation_gap,computed_gender_gap,LF_RATE,Labour force participation rate,NaN,NaN,PT_POP_SUB,Percentage of population in the same subgroup,"AGE=Y15T74; LABOUR_FORCE_STATUS=LF; SEX in [M, F]",Male OBS_VALUE - female OBS_VALUE,percentage points,"Gender gap in labour force participation rate,...",Higher = larger male advantage in labour-force...,Measures labour-force participation for ages 1...
3,pay_outcome,gender_wage_gap_median,gender_wage_gap,direct_indicator,GWP,Gender wage gap,NaN,NaN,PT_WG_SAL_M_D,Percentage of wages of men in the same decile,AGGREGATION_OPERATION=MEDIAN,Use OBS_VALUE directly,percent,"Median gender wage gap, expressed relative to ...",Higher = larger wage inequality / worse pay ou...,Unadjusted wage gap indicator. Should not be i...
4,work_structure_care,part_time_employment_gap_F_minus_M,part_time_employment_gap,computed_incidence_gap,EMP,Employment,NaN,NaN,PS,Persons,AGE=Y15T64; LABOUR_FORCE_STATUS=EMP; WORK_TIME...,For each sex: PT / (FT + PT) * 100; then femal...,percentage points,Gender gap in the incidence of part-time emplo...,Higher = women are more concentrated in part-t...,Measures the gender difference in part-time em...



Hypothesis mapping shape: (4, 10)


,hypothesis_id,hypothesis_name,research_hypothesis,null_hypothesis,alternative_hypothesis,independent_variable,dependent_variable,expected_direction,sample_note,interpretation_note
0,H1,STEM-to-leadership conversion,Countries with higher shares of women among te...,There is no association between women’s share ...,There is an association between women’s share ...,stem_women_share,women_senior_middle_mgmt_share,positive,Compare all available observations with OECD-m...,Leadership is measured with a senior/middle ma...
1,H2,STEM-to-pay equality,Countries with higher shares of women among te...,There is no association between women’s share ...,There is an association between women’s share ...,stem_women_share,gender_wage_gap_median,negative,Compare all available observations with OECD-m...,"The wage gap is unadjusted, so results are des..."
2,H3,Digital-skills gap and labour-market participa...,Countries with larger male advantages in youth...,There is no association between the youth codi...,There is an association between the youth codi...,coding_gap_youth_M_minus_F,labour_force_participation_gap_M_minus_F,positive,Compare all available observations with OECD-m...,Interpret cautiously because coding is measure...
3,H4,Work-structure bottleneck,Countries where women are more concentrated in...,There is no association between the gender gap...,There is an association between the gender gap...,part_time_employment_gap_F_minus_M,gender_wage_gap_median,positive,Compare all available observations with OECD-m...,Part-time gap is a work-structure proxy and do...


In [4]:
# df columns overview
pd.DataFrame({
    "column": df.columns,
    "dtype": [df[col].dtype for col in df.columns],
    "missing_count": [df[col].isna().sum() for col in df.columns],
    "missing_share": [df[col].isna().mean() for col in df.columns],
})

,column,dtype,missing_count,missing_share
0,REF_AREA,object,0,0.000000
1,is_oecd_member,bool,0,0.000000
2,is_aggregate_area,bool,0,0.000000
3,sample_scope,object,0,0.000000
4,stem_women_share,float64,15,0.250000
5,coding_gap_youth_M_minus_F,float64,21,0.350000
6,labour_force_participation_gap_M_minus_F,float64,2,0.033333
7,gender_wage_gap_median,float64,10,0.166667
8,part_time_employment_gap_F_minus_M,float64,8,0.133333
9,women_senior_middle_mgmt_share,float64,30,0.500000


## <a id="sample-structure"></a>Sample structure

This section checks the structure of the latest-wide dataset:

- how many rows belong to OECD member countries;
- how many rows are aggregate areas such as OECD, EU27, G7;
- how many rows are non-member / partner / candidate economies;
- whether Estonia is present and correctly labelled.

Aggregate areas are useful as possible reference points, but they should not be treated as countries in country-level EDA or hypothesis testing.

In [5]:
sample_scope_counts = (
    df["sample_scope"]
    .value_counts(dropna=False)
    .rename_axis("sample_scope")
    .reset_index(name="n_rows")
)

sample_scope_counts

,sample_scope,n_rows
0,OECD member,38
1,non-member / partner / candidate,14
2,aggregate area,8


In [6]:
# oecd / aggregate flags

scope_flags_summary = pd.DataFrame({
    "flag": ["is_oecd_member", "is_aggregate_area"],
    "true_count": [
        df["is_oecd_member"].sum(),
        df["is_aggregate_area"].sum(),
    ],
    "false_count": [
        (~df["is_oecd_member"]).sum(),
        (~df["is_aggregate_area"]).sum(),
    ],
})

scope_flags_summary

,flag,true_count,false_count
0,is_oecd_member,38,22
1,is_aggregate_area,8,52


In [7]:
# inspect aggregate areas
aggregates = df[df["is_aggregate_area"]].copy()

aggregates[["REF_AREA", "sample_scope"]]

,REF_AREA,sample_scope
17,EU19OECD,aggregate area
18,EU22OECD,aggregate area
19,EU25,aggregate area
20,EU27,aggregate area
21,EU28,aggregate area
24,G7,aggregate area
46,OECD,aggregate area
47,OECD_REP,aggregate area


In [8]:
# define working samples
df_all_countries = df[~df["is_aggregate_area"]].copy()

df_oecd = df[
    (df["is_oecd_member"]) & 
    (~df["is_aggregate_area"])
].copy()

df_aggregates = df[df["is_aggregate_area"]].copy()

print("All non-aggregate countries/economies:", df_all_countries.shape)
print("OECD member countries only:", df_oecd.shape)
print("Aggregate areas:", df_aggregates.shape)

All non-aggregate countries/economies: (52, 19)
OECD member countries only: (38, 19)
Aggregate areas: (8, 19)


In [9]:
# Estonia
estonia_row = df[df["REF_AREA"] == "EST"].copy()

estonia_row.T

,16
REF_AREA,EST
is_oecd_member,True
is_aggregate_area,False
sample_scope,OECD member
stem_women_share,41.545704
coding_gap_youth_M_minus_F,7.5205
labour_force_participation_gap_M_minus_F,4.924
gender_wage_gap_median,19.770206
part_time_employment_gap_F_minus_M,5.54288
women_senior_middle_mgmt_share,31.51


In [10]:
oecd_members_list = (
    df_oecd["REF_AREA"]
    .sort_values()
    .reset_index(drop=True)
)

oecd_members_list

0     AUS
1     AUT
2     BEL
3     CAN
4     CHE
5     CHL
6     COL
7     CRI
8     CZE
9     DEU
10    DNK
11    ESP
12    EST
13    FIN
14    FRA
15    GBR
16    GRC
17    HUN
18    IRL
19    ISL
20    ISR
21    ITA
22    JPN
23    KOR
24    LTU
25    LUX
26    LVA
27    MEX
28    NLD
29    NOR
30    NZL
31    POL
32    PRT
33    SVK
34    SVN
35    SWE
36    TUR
37    USA
Name: REF_AREA, dtype: object

In [11]:
# inspect non members
non_members = (
    df[df["sample_scope"] == "non-member / partner / candidate"]
    [["REF_AREA", "sample_scope"]]
    .sort_values("REF_AREA")
)

non_members

,REF_AREA,sample_scope
0,ARG,non-member / partner / candidate
4,BGR,non-member / partner / candidate
5,BRA,non-member / partner / candidate
11,CYP,non-member / partner / candidate
27,HRV,non-member / partner / candidate
29,IDN,non-member / partner / candidate
30,IND,non-member / partner / candidate
41,MKD,non-member / partner / candidate
42,MLT,non-member / partner / candidate
48,PER,non-member / partner / candidate


In [12]:
# vosual

fig = px.bar(
    sample_scope_counts,
    x="sample_scope",
    y="n_rows",
    text="n_rows",
    title="Dataset sample structure",
)

fig.update_layout(
    xaxis_title="Sample scope",
    yaxis_title="Number of rows",
)

fig.show()

In [13]:
# EU28 is reclassified as an aggregate area immediately after loading the data.
# This keeps every sample summary and working sample consistent.


In [14]:
# Broader non-aggregate sample:
# all individual countries/economies available in the API data.
df_all_countries = df[~df["is_aggregate_area"]].copy()

# Main sample for report:
# OECD member countries only, excluding aggregate areas.
df_main = df[
    (df["is_oecd_member"]) &
    (~df["is_aggregate_area"])
].copy()

# Alias, if we still want the explicit OECD name.
df_oecd = df_main.copy()

# Aggregate areas:
# can be used only as optional reference points, not as observations.
df_aggregates = df[df["is_aggregate_area"]].copy()

print("All non-aggregate countries/economies:", df_all_countries.shape)
print("Main OECD-member sample:", df_main.shape)
print("Aggregate areas:", df_aggregates.shape)

All non-aggregate countries/economies: (52, 19)
Main OECD-member sample: (38, 19)
Aggregate areas: (8, 19)


In [15]:
# verify sample scope counts after fix
sample_scope_counts = (
    df["sample_scope"]
    .value_counts(dropna=False)
    .rename_axis("sample_scope")
    .reset_index(name="n_rows")
)

sample_scope_counts

,sample_scope,n_rows
0,OECD member,38
1,non-member / partner / candidate,14
2,aggregate area,8


In [16]:
# main sample for report
df_main = df_oecd.copy()

# sensitivity check 
df_broader = df_all_countries.copy()

### Sample structure notes

The latest-wide dataset contains 60 rows: 38 OECD member countries, 15 non-member / partner / candidate economies, and 7 aggregate areas.

For the main EDA and all formal hypothesis tests, the analysis will use OECD member countries only and exclude aggregate areas. This keeps the comparison group aligned with the project scope and avoids treating aggregate regions such as OECD, EU27, or G7 as if they were individual countries.

The broader all-available non-aggregate sample may be used only as a sensitivity comparison during EDA. Aggregate areas may be used as optional reference points in visualizations, but they will not be included in country-level distributions, rankings, correlations, or regression models.

Estonia is included in the OECD-member sample and will be used as the focal country for benchmarking.

## <a id="missingness-in-the-latest-wide-dataset"></a>Missingness in the latest-wide dataset


This section checks missingness in the final latest-wide dataset.

The goal is to understand which indicators have complete or partial coverage in the main OECD-member sample and how this affects later EDA and hypothesis testing.

Because each hypothesis uses a pair of variables, formal tests will use pairwise complete observations rather than requiring every country to have all indicators.

In [17]:
# define indicator columns
indicator_cols = [
    "stem_women_share",
    "coding_gap_youth_M_minus_F",
    "labour_force_participation_gap_M_minus_F",
    "gender_wage_gap_median",
    "part_time_employment_gap_F_minus_M",
    "women_senior_middle_mgmt_share",
]

In [18]:
# missingness in main OECD sample
missingness_main = (
    df_main[indicator_cols]
    .isna()
    .sum()
    .reset_index()
)

missingness_main.columns = ["variable", "missing_count"]
missingness_main["n_rows"] = len(df_main)
missingness_main["missing_share"] = missingness_main["missing_count"] / len(df_main)

missingness_main

,variable,missing_count,n_rows,missing_share
0,stem_women_share,0,38,0.000000
1,coding_gap_youth_M_minus_F,6,38,0.157895
2,labour_force_participation_gap_M_minus_F,0,38,0.000000
3,gender_wage_gap_median,0,38,0.000000
4,part_time_employment_gap_F_minus_M,2,38,0.052632
5,women_senior_middle_mgmt_share,8,38,0.210526


In [19]:
# availability by country
country_availability_main = df_main[["REF_AREA"] + indicator_cols].copy()

country_availability_main["n_available_indicators"] = (
    country_availability_main[indicator_cols]
    .notna()
    .sum(axis=1)
)

country_availability_main["n_missing_indicators"] = (
    country_availability_main[indicator_cols]
    .isna()
    .sum(axis=1)
)

country_availability_main.sort_values(
    ["n_missing_indicators", "REF_AREA"],
    ascending=[False, True]
)

,REF_AREA,stem_women_share,coding_gap_youth_M_minus_F,labour_force_participation_gap_M_minus_F,gender_wage_gap_median,part_time_employment_gap_F_minus_M,women_senior_middle_mgmt_share,n_available_indicators,n_missing_indicators
10,CRI,37.315554,NaN,24.529,7.326732,11.129594,NaN,4,2
35,JPN,18.078102,NaN,11.747,20.650458,NaN,13.27,4,2
36,KOR,27.657623,4.376014,14.847,28.953678,NaN,NaN,4,2
45,NZL,43.052057,NaN,7.375,8.211731,15.699610,NaN,4,2
1,AUS,33.352506,NaN,7.650,10.671213,16.924777,39.39,5,1
6,CAN,35.630166,16.860000,7.674,16.497829,10.034182,NaN,5,1
8,CHL,20.991544,3.919409,17.976,13.593248,4.751383,NaN,5,1
9,COL,34.613893,4.206792,24.076,0.588235,5.208571,NaN,5,1
33,ISR,36.468030,NaN,4.100,20.846154,-10.772699,32.17,5,1
40,MEX,34.328589,3.934545,29.694,11.406423,8.799764,NaN,5,1


In [20]:
# comp[act summary
missing_indicators_summary = (
    country_availability_main["n_missing_indicators"]
    .value_counts()
    .sort_index()
    .rename_axis("n_missing_indicators")
    .reset_index(name="n_countries")
)

missing_indicators_summary

,n_missing_indicators,n_countries
0,0,26
1,1,8
2,2,4


In [21]:
# countries with incomplete indicator coverage
countries_with_missing = country_availability_main[
    country_availability_main["n_missing_indicators"] > 0
].sort_values(["n_missing_indicators", "REF_AREA"], ascending=[False, True])

countries_with_missing

,REF_AREA,stem_women_share,coding_gap_youth_M_minus_F,labour_force_participation_gap_M_minus_F,gender_wage_gap_median,part_time_employment_gap_F_minus_M,women_senior_middle_mgmt_share,n_available_indicators,n_missing_indicators
10,CRI,37.315554,NaN,24.529,7.326732,11.129594,NaN,4,2
35,JPN,18.078102,NaN,11.747,20.650458,NaN,13.27,4,2
36,KOR,27.657623,4.376014,14.847,28.953678,NaN,NaN,4,2
45,NZL,43.052057,NaN,7.375,8.211731,15.699610,NaN,4,2
1,AUS,33.352506,NaN,7.650,10.671213,16.924777,39.39,5,1
6,CAN,35.630166,16.860000,7.674,16.497829,10.034182,NaN,5,1
8,CHL,20.991544,3.919409,17.976,13.593248,4.751383,NaN,5,1
9,COL,34.613893,4.206792,24.076,0.588235,5.208571,NaN,5,1
33,ISR,36.468030,NaN,4.100,20.846154,-10.772699,32.17,5,1
40,MEX,34.328589,3.934545,29.694,11.406423,8.799764,NaN,5,1


In [22]:
# missingness by var visual


fig = px.bar(
    missingness_main,
    x="missing_count",      # по оси X — количество пропусков
    y="variable",           # по оси Y — индикаторы
    orientation="h",        # горизонтальный bar chart
    labels={
        "missing_count": "Missing values in OECD-member sample",
        "variable": "Indicator",
    },
    title="Missingness by indicator in main OECD sample",
)


fig.update_layout(
    xaxis_title="Missing values in OECD-member sample",
    yaxis_title="Indicator",
)

fig.show()

### Missingness notes

The main OECD-member sample contains 38 countries. Indicator coverage is mostly complete, but some countries have one or two missing indicators.

Missingness is handled through pairwise complete observations during hypothesis testing. This means that each hypothesis will use all countries that have data for both variables in that specific test pair, rather than restricting the analysis to countries with complete data across all six indicators.

This approach preserves sample size while keeping the missingness handling transparent.

## <a id="year-diagnostics"></a>Year diagnostics

The dataset uses the latest available value for each indicator. Because OECD indicators are updated on different schedules, the latest available years can differ across variables within the same country.

This section checks:

- which years are used for each indicator;
- how much latest-available years differ within each row;
- whether Estonia combines indicators from distant years;
- whether year mismatch may affect later interpretation.

The variable `latest_year_span` measures the distance between the newest and oldest indicator year available in each row.

In [23]:
year_cols = [
    "stem_women_share_year",
    "coding_gap_youth_M_minus_F_year",
    "labour_force_participation_gap_M_minus_F_year",
    "gender_wage_gap_median_year",
    "part_time_employment_gap_F_minus_M_year",
    "women_senior_middle_mgmt_share_year",
]

In [24]:
# year columns overview in main sample
year_overview_main = (
    df_main[year_cols]
    .agg(["min", "max", "median"])
    .T
    .reset_index()
    .rename(columns={"index": "year_column"})
)

year_overview_main

,year_column,min,max,median
0,stem_women_share_year,2023.0,2024.0,2024.0
1,coding_gap_youth_M_minus_F_year,2017.0,2025.0,2025.0
2,labour_force_participation_gap_M_minus_F_year,2025.0,2025.0,2025.0
3,gender_wage_gap_median_year,2022.0,2024.0,2024.0
4,part_time_employment_gap_F_minus_M_year,2025.0,2025.0,2025.0
5,women_senior_middle_mgmt_share_year,2022.0,2024.0,2023.0


In [25]:
# latest year span distribution
year_span_summary_main = (
    df_main["latest_year_span"]
    .describe()
    .reset_index()
    .rename(columns={"index": "statistic", "latest_year_span": "value"})
)

year_span_summary_main

,statistic,value
0,count,38.000000
1,mean,2.342105
2,std,1.320858
3,min,1.000000
4,25%,2.000000
5,50%,2.000000
6,75%,2.750000
7,max,8.000000


In [26]:
# countries with largest year span
year_span_by_country_main = (
    df_main[["REF_AREA", "latest_year_min", "latest_year_max", "latest_year_span"] + year_cols]
    .sort_values(["latest_year_span", "REF_AREA"], ascending=[False, True])
)

year_span_by_country_main.head(15)

,REF_AREA,latest_year_min,latest_year_max,latest_year_span,stem_women_share_year,coding_gap_youth_M_minus_F_year,labour_force_participation_gap_M_minus_F_year,gender_wage_gap_median_year,part_time_employment_gap_F_minus_M_year,women_senior_middle_mgmt_share_year
8,CHL,2017.0,2025.0,8.0,2024.0,2017.0,2025.0,2023.0,2025.0,NaN
25,GBR,2019.0,2025.0,6.0,2024.0,2019.0,2025.0,2024.0,2025.0,2023.0
32,ISL,2021.0,2025.0,4.0,2023.0,2021.0,2025.0,2022.0,2025.0,2023.0
3,BEL,2022.0,2025.0,3.0,2024.0,2025.0,2025.0,2022.0,2025.0,2023.0
6,CAN,2022.0,2025.0,3.0,2024.0,2022.0,2025.0,2024.0,2025.0,NaN
26,GRC,2022.0,2025.0,3.0,2024.0,2025.0,2025.0,2022.0,2025.0,2023.0
33,ISR,2022.0,2025.0,3.0,2024.0,NaN,2025.0,2022.0,2025.0,2023.0
35,JPN,2022.0,2025.0,3.0,2023.0,NaN,2025.0,2024.0,NaN,2022.0
38,LUX,2022.0,2025.0,3.0,2023.0,2025.0,2025.0,2022.0,2025.0,2023.0
50,PRT,2022.0,2025.0,3.0,2024.0,2025.0,2025.0,2022.0,2025.0,2023.0


In [27]:
# Estonia year diagnostics
estonia_years = df_main[df_main["REF_AREA"] == "EST"][
    ["REF_AREA", "latest_year_min", "latest_year_max", "latest_year_span"] + year_cols
]

estonia_years.T

,16
REF_AREA,EST
latest_year_min,2023.0
latest_year_max,2025.0
latest_year_span,2.0
stem_women_share_year,2024.0
coding_gap_youth_M_minus_F_year,2025.0
labour_force_participation_gap_M_minus_F_year,2025.0
gender_wage_gap_median_year,2024.0
part_time_employment_gap_F_minus_M_year,2025.0
women_senior_middle_mgmt_share_year,2023.0


In [28]:
# visual - year coverage by indicator

year_overview_plot = year_overview_main.copy()

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=year_overview_plot["min"],
    y=year_overview_plot["year_column"],
    mode="markers",
    name="Min year"
))

fig.add_trace(go.Scatter(
    x=year_overview_plot["median"],
    y=year_overview_plot["year_column"],
    mode="markers",
    name="Median year"
))

fig.add_trace(go.Scatter(
    x=year_overview_plot["max"],
    y=year_overview_plot["year_column"],
    mode="markers",
    name="Max year"
))

fig.update_layout(
    xaxis_title="Year",
    yaxis_title="Indicator year column",
    title="Latest available year range by indicator",
    height=400,
    width=800,
)
fig.update_traces(marker=dict(size=10))

fig.show()

### Pairwise timing alignment

The overall `latest_year_span` is useful for describing a country row, but each planned hypothesis uses a specific pair of indicators. The table below reports the year distance within the pairwise-complete OECD sample. It is a timing diagnostic only, not a hypothesis test.


In [29]:
pairwise_year_rows = []

for _, hypothesis in hypotheses.iterrows():
    x = hypothesis["independent_variable"]
    y = hypothesis["dependent_variable"]
    x_year = "{}_year".format(x)
    y_year = "{}_year".format(y)

    pair_df = df_main[["REF_AREA", x, y, x_year, y_year]].dropna().copy()
    pair_df["year_gap"] = (pair_df[x_year] - pair_df[y_year]).abs()

    max_gap = pair_df["year_gap"].max()
    max_gap_countries = ", ".join(
        pair_df.loc[pair_df["year_gap"] == max_gap, "REF_AREA"].sort_values()
    )
    estonia_gap = pair_df.loc[
        pair_df["REF_AREA"] == "EST", "year_gap"
    ].iloc[0]

    pairwise_year_rows.append({
        "hypothesis_id": hypothesis["hypothesis_id"],
        "hypothesis_name": hypothesis["hypothesis_name"],
        "n_pairwise_oecd": len(pair_df),
        "median_year_gap": pair_df["year_gap"].median(),
        "max_year_gap": max_gap,
        "countries_at_max_gap": max_gap_countries,
        "estonia_year_gap": estonia_gap,
    })

pairwise_year_diagnostics = pd.DataFrame(pairwise_year_rows)

pairwise_year_diagnostics[[
    "median_year_gap",
    "max_year_gap",
    "estonia_year_gap",
]] = pairwise_year_diagnostics[[
    "median_year_gap",
    "max_year_gap",
    "estonia_year_gap",
]].round(1)

pairwise_year_diagnostics


,hypothesis_id,hypothesis_name,n_pairwise_oecd,median_year_gap,max_year_gap,countries_at_max_gap,estonia_year_gap
0,H1,STEM-to-leadership conversion,30,1.0,1.0,"AUT, BEL, CHE, CZE, DEU, DNK, ESP, EST, FIN, G...",1.0
1,H2,STEM-to-pay equality,38,0.0,2.0,"BEL, GRC, ISR, PRT",0.0
2,H3,Digital-skills gap and labour-market participa...,32,0.0,8.0,CHL,0.0
3,H4,Work-structure bottleneck,36,1.0,3.0,"BEL, GRC, ISL, ISR, LUX, PRT",1.0


### Year diagnostics notes

The main OECD-member dataset uses latest-available values rather than a synchronized same-year cross-section.

Most indicators are recent: STEM values are from 2023–2024, labour-force participation and part-time employment values are from 2025, wage-gap values are from 2022–2024, and senior/middle management values are from 2022–2024. The coding indicator has the widest year range, from 2017 to 2025.

The median latest-year span across OECD member countries is 2 years. Estonia also has a 2-year span: the oldest latest-available indicator is women in senior/middle management from 2023, while the newest indicators are from 2025.

The pairwise timing table above is the relevant diagnostic for the planned hypothesis pairs. H1 is aligned within one year, H2 within two years, and H4 within three years. H3 has the widest mismatch: Chile has an 8-year gap and the United Kingdom a 6-year gap between youth coding and labour-force participation values.

Overall, the latest-year mismatch is documented but manageable for exploratory cross-country analysis. Formal hypothesis testing should keep the pairwise year-gap limitation explicit, especially for H3.

## <a id="descriptive-statistics"></a>Descriptive statistics

This section summarizes the core indicators in the main OECD-member sample.

The goal is to understand the central tendency, spread, and range of each variable before moving to distributions, Estonia benchmarking, and formal hypothesis testing.

The main descriptive statistics are calculated for `df_main`, which contains OECD member countries only and excludes aggregate areas.

In [30]:
# descriptive statistics for main sample
descriptive_stats_main = (
    df_main[indicator_cols]
    .describe()
    .T
    .reset_index()
    .rename(columns={"index": "variable"})
)

descriptive_stats_main

,variable,count,mean,std,min,25%,50%,75%,max
0,stem_women_share,38.0,33.781769,5.564671,18.078102,30.204921,34.246159,36.448664,43.291405
1,coding_gap_youth_M_minus_F,32.0,11.118349,6.116723,-0.371400,6.919050,10.050100,15.894850,28.849600
2,labour_force_participation_gap_M_minus_F,38.0,10.591158,7.206509,4.100000,6.177000,7.968500,11.414000,35.627000
3,gender_wage_gap_median,38.0,10.340741,6.739114,-8.677098,6.400965,10.744990,13.583403,28.953678
4,part_time_employment_gap_F_minus_M,36.0,10.881395,8.932622,-10.772699,4.987471,9.212901,16.005902,33.259281
5,women_senior_middle_mgmt_share,30.0,34.022667,7.093095,13.270000,31.607500,34.720000,38.657500,44.330000


In [31]:
# cleaner descriptive table

descriptive_stats_main = (
    df_main[indicator_cols]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .T
    .reset_index()
    .rename(columns={"index": "variable"})
)

descriptive_stats_main

,variable,count,mean,median,std,min,max
0,stem_women_share,38.0,33.781769,34.246159,5.564671,18.078102,43.291405
1,coding_gap_youth_M_minus_F,32.0,11.118349,10.050100,6.116723,-0.371400,28.849600
2,labour_force_participation_gap_M_minus_F,38.0,10.591158,7.968500,7.206509,4.100000,35.627000
3,gender_wage_gap_median,38.0,10.340741,10.744990,6.739114,-8.677098,28.953678
4,part_time_employment_gap_F_minus_M,36.0,10.881395,9.212901,8.932622,-10.772699,33.259281
5,women_senior_middle_mgmt_share,30.0,34.022667,34.720000,7.093095,13.270000,44.330000


In [32]:
metadata_cols = ["name", "final_unit", "direction", "layer"]

descriptive_stats_main = descriptive_stats_main.merge(
    data_dict[metadata_cols],
    left_on="variable",
    right_on="name",
    how="left"
).drop(columns=["name"])

descriptive_stats_main = descriptive_stats_main[
    [
        "variable",
        "layer",
        "count",
        "mean",
        "median",
        "std",
        "min",
        "max",
        "final_unit",
        "direction",
    ]
]

descriptive_stats_main

,variable,layer,count,mean,median,std,min,max,final_unit,direction
0,stem_women_share,education_pipeline,38.0,33.781769,34.246159,5.564671,18.078102,43.291405,percent,Higher = more women represented among tertiary...
1,coding_gap_youth_M_minus_F,digital_skills_pipeline,32.0,11.118349,10.050100,6.116723,-0.371400,28.849600,percentage points,Higher = larger male advantage in youth coding...
2,labour_force_participation_gap_M_minus_F,work_access,38.0,10.591158,7.968500,7.206509,4.100000,35.627000,percentage points,Higher = larger male advantage in labour-force...
3,gender_wage_gap_median,pay_outcome,38.0,10.340741,10.744990,6.739114,-8.677098,28.953678,percent,Higher = larger wage inequality / worse pay ou...
4,part_time_employment_gap_F_minus_M,work_structure_care,36.0,10.881395,9.212901,8.932622,-10.772699,33.259281,percentage points,Higher = women are more concentrated in part-t...
5,women_senior_middle_mgmt_share,leadership_proxy,30.0,34.022667,34.720000,7.093095,13.270000,44.330000,percent,Higher = more women represented in senior and ...


In [33]:
# round for readability
descriptive_stats_main_rounded = descriptive_stats_main.copy()

numeric_cols = ["count", "mean", "median", "std", "min", "max"]
descriptive_stats_main_rounded[numeric_cols] = (
    descriptive_stats_main_rounded[numeric_cols]
    .round(2)
)

descriptive_stats_main_rounded

,variable,layer,count,mean,median,std,min,max,final_unit,direction
0,stem_women_share,education_pipeline,38.0,33.78,34.25,5.56,18.08,43.29,percent,Higher = more women represented among tertiary...
1,coding_gap_youth_M_minus_F,digital_skills_pipeline,32.0,11.12,10.05,6.12,-0.37,28.85,percentage points,Higher = larger male advantage in youth coding...
2,labour_force_participation_gap_M_minus_F,work_access,38.0,10.59,7.97,7.21,4.10,35.63,percentage points,Higher = larger male advantage in labour-force...
3,gender_wage_gap_median,pay_outcome,38.0,10.34,10.74,6.74,-8.68,28.95,percent,Higher = larger wage inequality / worse pay ou...
4,part_time_employment_gap_F_minus_M,work_structure_care,36.0,10.88,9.21,8.93,-10.77,33.26,percentage points,Higher = women are more concentrated in part-t...
5,women_senior_middle_mgmt_share,leadership_proxy,30.0,34.02,34.72,7.09,13.27,44.33,percent,Higher = more women represented in senior and ...


In [34]:
# comparison with non-oecd
def build_descriptive_stats(sample_df, sample_name, indicator_cols, data_dict):
    stats = (
        sample_df[indicator_cols]
        .agg(["count", "mean", "median", "std", "min", "max"])
        .T
        .reset_index()
        .rename(columns={"index": "variable"})
    )

    stats = stats.merge(
        data_dict[["name", "layer", "final_unit", "direction"]],
        left_on="variable",
        right_on="name",
        how="left"
    ).drop(columns=["name"])

    stats["sample"] = sample_name

    stats = stats[
        [
            "sample",
            "variable",
            "layer",
            "count",
            "mean",
            "median",
            "std",
            "min",
            "max",
            "final_unit",
            "direction",
        ]
    ]

    return stats

descriptive_stats_oecd = build_descriptive_stats(
    df_main,
    "oecd_members",
    indicator_cols,
    data_dict
)

descriptive_stats_broader = build_descriptive_stats(
    df_all_countries,
    "all_non_aggregate",
    indicator_cols,
    data_dict
)

descriptive_stats_comparison = pd.concat(
    [descriptive_stats_oecd, descriptive_stats_broader],
    ignore_index=True
)

descriptive_stats_comparison_rounded = descriptive_stats_comparison.copy()

numeric_cols = ["count", "mean", "median", "std", "min", "max"]
descriptive_stats_comparison_rounded[numeric_cols] = (
    descriptive_stats_comparison_rounded[numeric_cols].round(2)
)

descriptive_stats_comparison_rounded

,sample,variable,layer,count,mean,median,std,min,max,final_unit,direction
0,oecd_members,stem_women_share,education_pipeline,38.0,33.78,34.25,5.56,18.08,43.29,percent,Higher = more women represented among tertiary...
1,oecd_members,coding_gap_youth_M_minus_F,digital_skills_pipeline,32.0,11.12,10.05,6.12,-0.37,28.85,percentage points,Higher = larger male advantage in youth coding...
2,oecd_members,labour_force_participation_gap_M_minus_F,work_access,38.0,10.59,7.97,7.21,4.10,35.63,percentage points,Higher = larger male advantage in labour-force...
3,oecd_members,gender_wage_gap_median,pay_outcome,38.0,10.34,10.74,6.74,-8.68,28.95,percent,Higher = larger wage inequality / worse pay ou...
4,oecd_members,part_time_employment_gap_F_minus_M,work_structure_care,36.0,10.88,9.21,8.93,-10.77,33.26,percentage points,Higher = women are more concentrated in part-t...
5,oecd_members,women_senior_middle_mgmt_share,leadership_proxy,30.0,34.02,34.72,7.09,13.27,44.33,percent,Higher = more women represented in senior and ...
6,all_non_aggregate,stem_women_share,education_pipeline,43.0,34.36,34.47,5.85,18.08,47.81,percent,Higher = more women represented among tertiary...
7,all_non_aggregate,coding_gap_youth_M_minus_F,digital_skills_pipeline,36.0,10.55,9.94,6.13,-0.37,28.85,percentage points,Higher = larger male advantage in youth coding...
8,all_non_aggregate,labour_force_participation_gap_M_minus_F,work_access,52.0,12.59,9.60,8.47,4.10,44.09,percentage points,Higher = larger male advantage in labour-force...
9,all_non_aggregate,gender_wage_gap_median,pay_outcome,47.0,10.69,10.67,7.49,-8.68,35.26,percent,Higher = larger wage inequality / worse pay ou...


In [35]:
median_comparison = descriptive_stats_comparison_rounded[
    ["sample", "variable", "median", "final_unit"]
].copy()

median_comparison

,sample,variable,median,final_unit
0,oecd_members,stem_women_share,34.25,percent
1,oecd_members,coding_gap_youth_M_minus_F,10.05,percentage points
2,oecd_members,labour_force_participation_gap_M_minus_F,7.97,percentage points
3,oecd_members,gender_wage_gap_median,10.74,percent
4,oecd_members,part_time_employment_gap_F_minus_M,9.21,percentage points
5,oecd_members,women_senior_middle_mgmt_share,34.72,percent
6,all_non_aggregate,stem_women_share,34.47,percent
7,all_non_aggregate,coding_gap_youth_M_minus_F,9.94,percentage points
8,all_non_aggregate,labour_force_participation_gap_M_minus_F,9.60,percentage points
9,all_non_aggregate,gender_wage_gap_median,10.67,percent


In [36]:
# Visual comparison of country-level medians.
# The indicators use different units, so compare values within each indicator only.
pivot_medians = (
    median_comparison
    .pivot(index="variable", columns="sample", values="median")
)

pivot_medians_reset = pivot_medians.reset_index()

fig = px.bar(
    pivot_medians_reset,
    x=["oecd_members", "all_non_aggregate"],
    y="variable",
    orientation="h",
    labels={
        "variable": "Indicator",
        "value": "Median value",
        "sample": "Sample",
    },
    title="Median indicator values: OECD members vs all non-aggregate countries/economies",
)

fig.update_layout(
    xaxis_title="Median value",
    yaxis_title="Indicator",
    barmode="group",
)

fig.show()


In [37]:
# median difference is better answers the question
# does the sample extension change the picture or not?
median_diff = pivot_medians.copy()

median_diff["difference_all_minus_oecd"] = (
    median_diff["all_non_aggregate"] - median_diff["oecd_members"]
)

median_diff = median_diff.reset_index()

median_diff

sample,variable,all_non_aggregate,oecd_members,difference_all_minus_oecd
0,coding_gap_youth_M_minus_F,9.94,10.05,-0.11
1,gender_wage_gap_median,10.67,10.74,-0.07
2,labour_force_participation_gap_M_minus_F,9.60,7.97,1.63
3,part_time_employment_gap_F_minus_M,7.22,9.21,-1.99
4,stem_women_share,34.47,34.25,0.22
5,women_senior_middle_mgmt_share,34.72,34.72,0.00


In [38]:
fig = px.bar(
    median_diff,
    x="difference_all_minus_oecd",   
    y="variable",                    
    orientation="h",                
    labels={
        "variable": "Indicator",
        "difference_all_minus_oecd": "Median difference: all non-aggregate minus OECD",
    },
    title="How much does the median change when using the broader sample?",
)

fig.add_vline(
    x=0,
    line_width=1,
    line_color="black"
)

fig.update_layout(
    xaxis_title="Median difference: all non-aggregate minus OECD",
    yaxis_title="Indicator",
)

fig.show()

In [39]:
descriptive_stats_oecd.to_csv(DATA_PATH + "descriptive_stats_oecd.csv", index=False)
descriptive_stats_comparison_rounded.to_csv(DATA_PATH + "descriptive_stats_comparison.csv", index=False)

### Descriptive statistics notes

The main OECD-member sample contains 38 countries, but available observations vary by indicator. The most limited indicator is `women_senior_middle_mgmt_share`, with 30 available OECD observations, followed by `coding_gap_youth_M_minus_F`, with 32 observations.

Across OECD members, women represent on average around one third of tertiary STEM graduates. The average share of women in senior and middle management positions is also around one third, but this variable has a wider range, from 13.27% to 44.33%.

The gap indicators show different scales. The average youth coding gap is around 11 percentage points in favour of men. The average labour-force participation gap is around 10.6 percentage points, while the mean country-level value of the median gender wage gap indicator is around 10.3%.

Some variables contain unusual values, including negative values for the median gender wage gap and for the part-time employment gap. These cases should be reviewed later in the outlier section before formal hypothesis testing.

## <a id="indicator-distributions"></a>Indicator distributions

This section explores the distribution of each core indicator in the main OECD-member sample.

The goal is to understand:

- the shape of each distribution;
- the spread and possible skewness of values;
- whether unusual values or potential outliers are present;
- where Estonia sits visually within each indicator distribution.

These plots are exploratory and descriptive. They do not test hypotheses formally.

In [40]:
indicator_cols = [
    "stem_women_share",
    "coding_gap_youth_M_minus_F",
    "labour_force_participation_gap_M_minus_F",
    "gender_wage_gap_median",
    "part_time_employment_gap_F_minus_M",
    "women_senior_middle_mgmt_share",
]

In [41]:
# Histogram for each indicator

# Long-format: indicator, value
df_long = (
    df_main[indicator_cols]
    .melt(
        value_name="value",
        var_name="indicator"
    )
    .dropna(subset=["value"])
)

fig = px.histogram(
    df_long,
    x="value",
    facet_col="indicator",       
    facet_col_wrap=3,            
    nbins=10,
    labels={
        "value": "Indicator value",
        "indicator": "Indicator",
    },
    title="Distribution of indicators in OECD-member sample",
)

fig.update_yaxes(title_text="Number of OECD countries")
fig.update_xaxes(matches=None)  

fig.update_layout(
    height=800,   
    width=1100,
    showlegend=False,
)

fig.show()

In [42]:
# box plots

df_long = (
    df_main[indicator_cols]
    .melt(var_name="indicator", value_name="value")
    .dropna(subset=["value"])
)

fig = px.box(
    df_long,
    x="indicator",
    y="value",
    facet_col="indicator",
    facet_col_wrap=3,
    points="outliers",
    title="Distribution of indicators in OECD-member sample",
    labels={
        "value": "Indicator value",
        "indicator": "Indicator",
    },
)

fig.update_layout(
    showlegend=False,
    height=800,
    width=1100,
)

fig.update_xaxes(matches=None, showticklabels=False)
fig.show()

In [43]:
# Estonia marker on a simple strip plot

for col in indicator_cols:
    clean = df_main[["REF_AREA", col]].dropna().copy()
    clean = clean.rename(columns={col: "value"})

    # est value
    est_value = clean.loc[clean["REF_AREA"] == "EST", "value"].iloc[0]

    fig = go.Figure()

    # all countries
    fig.add_trace(go.Scatter(
        x=clean["value"],
        y=[0] * len(clean),
        mode="markers",
        marker=dict(color="rgba(31, 119, 180, 0.8)", size=7),
        showlegend=False
    ))

    # Estonia
    fig.add_trace(go.Scatter(
        x=[est_value],
        y=[0],
        mode="markers+text",
        marker=dict(color="orange", size=10),
        text=["EST"],
        textposition="top right",
        textfont=dict(size=10),
        showlegend=False
    ))

    fig.update_yaxes(
        visible=False,
        showticklabels=False
    )

    fig.update_layout(
        xaxis_title=col,
        title=f"{col}: OECD-member distribution with Estonia highlighted",
        height=250,
        width=800,
        margin=dict(l=40, r=20, t=60, b=40),
        plot_bgcolor="white"
    )

    fig.show()

In [44]:
# distribution_extremes / min_max_summary

min_max_summary = []

for col in indicator_cols:
    clean = df_main[["REF_AREA", col]].dropna().copy()
    
    min_row = clean.loc[clean[col].idxmin()]
    max_row = clean.loc[clean[col].idxmax()]
    est_row = clean[clean["REF_AREA"] == "EST"].iloc[0]

    min_max_summary.append({
        "variable": col,
        "min_country": min_row["REF_AREA"],
        "min_value": min_row[col],
        "max_country": max_row["REF_AREA"],
        "max_value": max_row[col],
        "estonia_value": est_row[col],
    })

min_max_summary = pd.DataFrame(min_max_summary)
min_max_summary

,variable,min_country,min_value,max_country,max_value,estonia_value
0,stem_women_share,JPN,18.078102,ISL,43.291405,41.545704
1,coding_gap_youth_M_minus_F,GRC,-0.371400,IRL,28.849600,7.520500
2,labour_force_participation_gap_M_minus_F,ISR,4.100000,TUR,35.627000,4.924000
3,gender_wage_gap_median,LUX,-8.677098,KOR,28.953678,19.770206
4,part_time_employment_gap_F_minus_M,ISR,-10.772699,NLD,33.259281,5.542880
5,women_senior_middle_mgmt_share,JPN,13.270000,USA,44.330000,31.510000


### Distribution notes

The indicator distributions show different levels of cross-country variation in the OECD-member sample.

`stem_women_share` is relatively concentrated around the low-to-mid 30% range. Estonia is positioned near the high end of the OECD distribution, with Japan at the low end and Iceland at the high end.

`coding_gap_youth_M_minus_F` varies more widely and includes one country where the gap is slightly negative, meaning the female value is marginally higher than the male value. Estonia is below the OECD median, indicating a smaller male advantage in youth coding than in many OECD countries.

`labour_force_participation_gap_M_minus_F` is right-skewed: most OECD countries have relatively moderate gaps, while a few countries have much larger male advantages. Estonia is near the low end of the distribution.

`gender_wage_gap_median` has a broad range, including one negative value and several high-gap countries. Estonia is positioned toward the higher end of the OECD distribution, meaning its median gender wage gap is relatively large compared with many OECD members.

`part_time_employment_gap_F_minus_M` also varies substantially. Most countries show higher part-time incidence among women than men, but at least one country has a negative gap. Estonia is below the OECD median, suggesting a smaller gender gap in part-time employment than in many OECD countries.

`women_senior_middle_mgmt_share` is concentrated around roughly one third, but with some low-end cases. Estonia is slightly below the OECD median for this indicator.

### Indicator extremes and plausibility check

The min/max summary identifies the countries at the lower and upper ends of each indicator distribution, together with Estonia’s value.

This is not a formal outlier detection step. Since the dataset is based on OECD / OECD-linked official indicators, extreme values are not treated as errors by default. Instead, they are reviewed as potentially meaningful country-level cases or as values that may require careful interpretation.

The extreme values help identify indicators with wider cross-country variation, such as labour-force participation gaps, part-time employment gaps, wage gaps, and senior/middle management representation.

## <a id="estonia-benchmark"></a>Estonia benchmark

This section benchmarks Estonia against the OECD-member sample for each core indicator.

For each variable, the benchmark table compares Estonia’s latest available value with the OECD-member mean and median, and calculates Estonia’s rank and percentile position.

Interpretation depends on indicator direction:

- for representation indicators, higher values usually indicate stronger representation of women;
- for gap indicators, higher values usually indicate larger inequality or larger male advantage;
- for `part_time_employment_gap_F_minus_M`, higher values indicate that women are more concentrated in part-time employment relative to men.

This benchmark is descriptive and does not test hypotheses.

In [45]:
# helper metadata

indicator_metadata = (
    data_dict[["name", "layer", "final_unit", "direction"]]
    .drop_duplicates(subset=["name"])
    .set_index("name")
)

In [46]:
# define direction groups
# to understand where higher = better / worse / structural

higher_is_better = [
    "stem_women_share",
    "women_senior_middle_mgmt_share",
]

higher_is_worse = [
    "coding_gap_youth_M_minus_F",
    "labour_force_participation_gap_M_minus_F",
    "gender_wage_gap_median",
]

higher_is_structural_gap = [
    "part_time_employment_gap_F_minus_M",
]

In [47]:
# build Estonia benchmark table
benchmark_rows = []

for col in indicator_cols:
    clean = df_main[["REF_AREA", col]].dropna().copy()
    
    est_series = clean.loc[clean["REF_AREA"] == "EST", col]
    
    if est_series.empty:
        continue
    
    est_value = est_series.iloc[0]
    
    oecd_mean = clean[col].mean()
    oecd_median = clean[col].median()
    oecd_min = clean[col].min()
    oecd_max = clean[col].max()
    n_countries = len(clean)

    # Rank: 1 = highest value
    rank_high_to_low = clean[col].rank(ascending=False, method="min")
    est_rank_high_to_low = int(rank_high_to_low.loc[clean["REF_AREA"] == "EST"].iloc[0])

    # Rank: 1 = lowest value
    rank_low_to_high = clean[col].rank(ascending=True, method="min")
    est_rank_low_to_high = int(rank_low_to_high.loc[clean["REF_AREA"] == "EST"].iloc[0])

    # Percentile: share of countries with values <= Estonia's value
    percentile_low_to_high = (clean[col] <= est_value).mean() * 100

    if col in higher_is_better:
        interpretation_direction = "higher = better representation"
        directional_rank = est_rank_high_to_low
        directional_rank_label = "rank_high_to_low"
    elif col in higher_is_worse:
        interpretation_direction = "higher = worse / larger inequality"
        directional_rank = est_rank_low_to_high
        directional_rank_label = "rank_low_to_high"
    elif col in higher_is_structural_gap:
        interpretation_direction = "higher = larger female part-time concentration"
        directional_rank = est_rank_low_to_high
        directional_rank_label = "rank_low_to_high"
    else:
        interpretation_direction = "check direction"
        directional_rank = np.nan
        directional_rank_label = np.nan

    benchmark_rows.append({
        "variable": col,
        "layer": indicator_metadata.loc[col, "layer"],
        "estonia_value": est_value,
        "oecd_mean": oecd_mean,
        "oecd_median": oecd_median,
        "difference_from_oecd_median": est_value - oecd_median,
        "oecd_min": oecd_min,
        "oecd_max": oecd_max,
        "n_countries": n_countries,
        "estonia_rank_high_to_low": est_rank_high_to_low,
        "estonia_rank_low_to_high": est_rank_low_to_high,
        "estonia_percentile_low_to_high": percentile_low_to_high,
        "directional_rank": directional_rank,
        "directional_rank_label": directional_rank_label,
        "final_unit": indicator_metadata.loc[col, "final_unit"],
        "interpretation_direction": interpretation_direction,
        "metadata_direction": indicator_metadata.loc[col, "direction"],
    })

estonia_benchmark = pd.DataFrame(benchmark_rows)
estonia_benchmark

,variable,layer,estonia_value,oecd_mean,oecd_median,difference_from_oecd_median,oecd_min,oecd_max,n_countries,estonia_rank_high_to_low,estonia_rank_low_to_high,estonia_percentile_low_to_high,directional_rank,directional_rank_label,final_unit,interpretation_direction,metadata_direction
0,stem_women_share,education_pipeline,41.545704,33.781769,34.246159,7.299545,18.078102,43.291405,38,4,35,92.105263,4,rank_high_to_low,percent,higher = better representation,Higher = more women represented among tertiary...
1,coding_gap_youth_M_minus_F,digital_skills_pipeline,7.520500,11.118349,10.050100,-2.529600,-0.371400,28.849600,32,23,10,31.250000,10,rank_low_to_high,percentage points,higher = worse / larger inequality,Higher = larger male advantage in youth coding...
2,labour_force_participation_gap_M_minus_F,work_access,4.924000,10.591158,7.968500,-3.044500,4.100000,35.627000,38,34,5,13.157895,5,rank_low_to_high,percentage points,higher = worse / larger inequality,Higher = larger male advantage in labour-force...
3,gender_wage_gap_median,pay_outcome,19.770206,10.340741,10.744990,9.025216,-8.677098,28.953678,38,4,35,92.105263,35,rank_low_to_high,percent,higher = worse / larger inequality,Higher = larger wage inequality / worse pay ou...
4,part_time_employment_gap_F_minus_M,work_structure_care,5.542880,10.881395,9.212901,-3.670021,-10.772699,33.259281,36,24,13,36.111111,13,rank_low_to_high,percentage points,higher = larger female part-time concentration,Higher = women are more concentrated in part-t...
5,women_senior_middle_mgmt_share,leadership_proxy,31.510000,34.022667,34.720000,-3.210000,13.270000,44.330000,30,23,8,26.666667,23,rank_high_to_low,percent,higher = better representation,Higher = more women represented in senior and ...


In [48]:
# rounded and readable table
estonia_benchmark_rounded = estonia_benchmark.copy()

numeric_cols = [
    "estonia_value",
    "oecd_mean",
    "oecd_median",
    "difference_from_oecd_median",
    "oecd_min",
    "oecd_max",
    "estonia_percentile_low_to_high",
]

estonia_benchmark_rounded[numeric_cols] = (
    estonia_benchmark_rounded[numeric_cols]
    .round(2)
)

estonia_benchmark_rounded

,variable,layer,estonia_value,oecd_mean,oecd_median,difference_from_oecd_median,oecd_min,oecd_max,n_countries,estonia_rank_high_to_low,estonia_rank_low_to_high,estonia_percentile_low_to_high,directional_rank,directional_rank_label,final_unit,interpretation_direction,metadata_direction
0,stem_women_share,education_pipeline,41.55,33.78,34.25,7.30,18.08,43.29,38,4,35,92.11,4,rank_high_to_low,percent,higher = better representation,Higher = more women represented among tertiary...
1,coding_gap_youth_M_minus_F,digital_skills_pipeline,7.52,11.12,10.05,-2.53,-0.37,28.85,32,23,10,31.25,10,rank_low_to_high,percentage points,higher = worse / larger inequality,Higher = larger male advantage in youth coding...
2,labour_force_participation_gap_M_minus_F,work_access,4.92,10.59,7.97,-3.04,4.10,35.63,38,34,5,13.16,5,rank_low_to_high,percentage points,higher = worse / larger inequality,Higher = larger male advantage in labour-force...
3,gender_wage_gap_median,pay_outcome,19.77,10.34,10.74,9.03,-8.68,28.95,38,4,35,92.11,35,rank_low_to_high,percent,higher = worse / larger inequality,Higher = larger wage inequality / worse pay ou...
4,part_time_employment_gap_F_minus_M,work_structure_care,5.54,10.88,9.21,-3.67,-10.77,33.26,36,24,13,36.11,13,rank_low_to_high,percentage points,higher = larger female part-time concentration,Higher = women are more concentrated in part-t...
5,women_senior_middle_mgmt_share,leadership_proxy,31.51,34.02,34.72,-3.21,13.27,44.33,30,23,8,26.67,23,rank_high_to_low,percent,higher = better representation,Higher = more women represented in senior and ...


In [49]:
estonia_benchmark_rounded.to_csv(
    DATA_PATH + "estonia_benchmark_oecd.csv",
    index=False
)

In [50]:
# Estonia vs OECD median

estonia_vs_median = estonia_benchmark_rounded[
    [
        "variable",
        "estonia_value",
        "oecd_median",
        "difference_from_oecd_median",
        "final_unit",
        "interpretation_direction",
    ]
].copy()

estonia_vs_median

,variable,estonia_value,oecd_median,difference_from_oecd_median,final_unit,interpretation_direction
0,stem_women_share,41.55,34.25,7.30,percent,higher = better representation
1,coding_gap_youth_M_minus_F,7.52,10.05,-2.53,percentage points,higher = worse / larger inequality
2,labour_force_participation_gap_M_minus_F,4.92,7.97,-3.04,percentage points,higher = worse / larger inequality
3,gender_wage_gap_median,19.77,10.74,9.03,percent,higher = worse / larger inequality
4,part_time_employment_gap_F_minus_M,5.54,9.21,-3.67,percentage points,higher = larger female part-time concentration
5,women_senior_middle_mgmt_share,31.51,34.72,-3.21,percent,higher = better representation


In [51]:
# visual
# positive difference isn't always good. Therefore, in a report, it's best to either
# separate the visual with the signs explained
# or use a direction-aware normalized score
# this visual is EDA only!!!

fig = px.bar(
    estonia_vs_median,
    x="difference_from_oecd_median",  
    y="variable",                      
    orientation="h",
    labels={
        "variable": "Indicator",
        "difference_from_oecd_median": "Estonia - OECD median",
    },
    title="Estonia compared with OECD median by indicator",
)


fig.add_vline(
    x=0,
    line_width=1,
    line_color="black"
)

fig.update_layout(
    xaxis_title="Estonia - OECD median",
    yaxis_title="Indicator",
)

fig.show()

### Estonia benchmark notes

Estonia’s position differs substantially across the six indicators, which supports the project’s “conversion gap” framing.

Estonia is well above the OECD median in women’s share among tertiary STEM graduates: 41.55% compared with an OECD median of 34.25%. This places Estonia near the high end of the OECD distribution for the education pipeline indicator.

Estonia also performs relatively well on some labour-market access and work-structure indicators. Its youth coding gender gap is below the OECD median, meaning the male advantage in youth coding activity is smaller than in many OECD countries. Estonia’s labour-force participation gap is also below the OECD median, placing it among the countries with relatively smaller gender gaps in labour-force participation. The part-time employment gap is below the OECD median as well, suggesting that women are less disproportionately concentrated in part-time employment than in many OECD countries.

However, Estonia’s median gender wage gap is substantially above the OECD median: 19.77% compared with 10.74%. Because higher values of this indicator mean worse pay outcomes for women, this is one of Estonia’s weaker positions in the benchmark.

Estonia is also below the OECD median in women’s representation in senior and middle management positions: 31.51% compared with 34.72%. This is a descriptive contrast between country-level indicators; it does not establish whether a cohort of STEM graduates progresses into senior and middle management.

Overall, the benchmark suggests a mixed profile: Estonia appears relatively strong in STEM education and some labour-market access indicators, but weaker in pay equality and leadership representation. This makes Estonia a relevant focal case for examining cross-country associations between STEM and digital participation indicators and labour-market power outcomes.


## <a id="exploratory-relationship-plots"></a>Exploratory relationship plots

This section visually inspects the variable pairs connected to the working hypotheses.

These plots are exploratory. They are used to understand the shape of the relationships, Estonia’s position, and possible influential country-level cases before formal statistical testing.

No p-values or formal hypothesis conclusions are reported in this section. Formal tests are reserved for `02_hypothesis_testing.ipynb`.

In [52]:
# hypothesis pairs

relationship_pairs = pd.DataFrame([
    {
        "hypothesis": "H1",
        "name": "STEM-to-leadership conversion",
        "x": "stem_women_share",
        "y": "women_senior_middle_mgmt_share",
        "expected_direction": "positive",
    },
    {
        "hypothesis": "H2",
        "name": "STEM-to-pay equality",
        "x": "stem_women_share",
        "y": "gender_wage_gap_median",
        "expected_direction": "negative",
    },
    {
        "hypothesis": "H3",
        "name": "Digital-skills gap and labour-market participation",
        "x": "coding_gap_youth_M_minus_F",
        "y": "labour_force_participation_gap_M_minus_F",
        "expected_direction": "positive",
    },
    {
        "hypothesis": "H4",
        "name": "Work-structure bottleneck",
        "x": "part_time_employment_gap_F_minus_M",
        "y": "gender_wage_gap_median",
        "expected_direction": "positive",
    },
])

relationship_pairs

,hypothesis,name,x,y,expected_direction
0,H1,STEM-to-leadership conversion,stem_women_share,women_senior_middle_mgmt_share,positive
1,H2,STEM-to-pay equality,stem_women_share,gender_wage_gap_median,negative
2,H3,Digital-skills gap and labour-market participa...,coding_gap_youth_M_minus_F,labour_force_participation_gap_M_minus_F,positive
3,H4,Work-structure bottleneck,part_time_employment_gap_F_minus_M,gender_wage_gap_median,positive


In [53]:
# check available observations per pair
relationship_coverage_rows = []

for _, row in relationship_pairs.iterrows():
    clean = df_main[["REF_AREA", row["x"], row["y"]]].dropna()
    
    relationship_coverage_rows.append({
        "hypothesis": row["hypothesis"],
        "name": row["name"],
        "x": row["x"],
        "y": row["y"],
        "n_oecd_countries": len(clean),
        "estonia_included": "EST" in clean["REF_AREA"].values,
    })

relationship_coverage = pd.DataFrame(relationship_coverage_rows)
relationship_coverage

,hypothesis,name,x,y,n_oecd_countries,estonia_included
0,H1,STEM-to-leadership conversion,stem_women_share,women_senior_middle_mgmt_share,30,True
1,H2,STEM-to-pay equality,stem_women_share,gender_wage_gap_median,38,True
2,H3,Digital-skills gap and labour-market participa...,coding_gap_youth_M_minus_F,labour_force_participation_gap_M_minus_F,32,True
3,H4,Work-structure bottleneck,part_time_employment_gap_F_minus_M,gender_wage_gap_median,36,True


In [54]:
def plot_relationship(df_sample, x, y, title):
    plot_df = df_sample[["REF_AREA", x, y]].dropna().copy()
    plot_df["label"] = np.where(plot_df["REF_AREA"] == "EST", "EST", "")
    plot_df["country_type"] = np.where(plot_df["REF_AREA"] == "EST", "Estonia", "Other OECD countries")

    fig = px.scatter(
        plot_df,
        x=x,
        y=y,
        text="label",
        hover_name="REF_AREA",
        color="country_type",
        title=title,
    )

    fig.update_traces(textposition="top center")
    fig.update_layout(
        xaxis_title=x,
        yaxis_title=y,
        legend_title_text="Country",
    )

    fig.show()

for _, row in relationship_pairs.iterrows():
    plot_relationship(
        df_main,
        row["x"],
        row["y"],
        f'{row["hypothesis"]}: {row["name"]}'
    )

In [55]:
relationship_coverage.to_csv(
    DATA_PATH + "eda_relationship_plot_coverage.csv",
    index=False
)

### Exploratory relationship notes

The hypothesis-related scatterplots provide a first visual inspection of the relationship structure before formal testing.

For H1, Estonia appears high on women’s share among tertiary STEM graduates but not high on women’s representation in senior and middle management. This is a descriptive country profile that motivates the project’s “conversion gap” framing; it does not establish a cohort-level conversion process.

For H2, Estonia combines a high STEM women share with a high median gender wage gap. This is especially relevant because the expected direction for this hypothesis is negative: if STEM participation translated directly into pay equality, countries with higher STEM women share would be expected to have lower wage gaps. The visual pattern does not suggest a simple relationship.

For H3, the relationship between the youth coding gap and the labour-force participation gap does not look straightforward. Estonia is positioned with a moderate-to-low youth coding gap and a low labour-force participation gap, but other countries show different combinations. This is an exploratory country-level comparison: youth coding refers to ages 16–24, while labour-force participation refers to ages 15–74, so the indicators do not describe the same population.

For H4, the part-time employment gap and median gender wage gap do not show an obvious simple positive pattern in the scatterplot. Estonia has a below-median part-time employment gap but an above-median wage gap, suggesting that wage inequality cannot be explained by part-time concentration alone.

Overall, the exploratory relationship plots suggest that the conversion from STEM/digital participation into labour-market power is not automatic or one-dimensional. Formal correlation and regression tests will be conducted in the hypothesis testing notebook.


## <a id="indicator-extremes-top-and-bottom-countries"></a>Indicator extremes: top and bottom countries

This section identifies the countries at the lower and upper ends of each indicator distribution.

This is not a formal outlier detection step. The goal is to understand which countries define the observed range of each variable and whether any extreme values require careful interpretation before hypothesis testing.

Extreme values are not removed automatically.

In [56]:
top_bottom_rows = []

for col in indicator_cols:
    clean = df_main[["REF_AREA", col]].dropna().copy()
    
    bottom = clean.sort_values(col, ascending=True).head(5)
    top = clean.sort_values(col, ascending=False).head(5)
    
    for _, row in bottom.iterrows():
        top_bottom_rows.append({
            "variable": col,
            "position": "bottom_5",
            "REF_AREA": row["REF_AREA"],
            "value": row[col],
        })
        
    for _, row in top.iterrows():
        top_bottom_rows.append({
            "variable": col,
            "position": "top_5",
            "REF_AREA": row["REF_AREA"],
            "value": row[col],
        })

top_bottom_countries = pd.DataFrame(top_bottom_rows)
top_bottom_countries

,variable,position,REF_AREA,value
0,stem_women_share,bottom_5,JPN,18.078102
1,stem_women_share,bottom_5,CHL,20.991544
2,stem_women_share,bottom_5,CHE,26.050484
3,stem_women_share,bottom_5,ESP,27.498438
4,stem_women_share,bottom_5,KOR,27.657623
5,stem_women_share,top_5,ISL,43.291405
6,stem_women_share,top_5,NZL,43.052057
7,stem_women_share,top_5,GRC,41.682659
8,stem_women_share,top_5,EST,41.545704
9,stem_women_share,top_5,POL,40.232751


In [57]:
# + metadata
top_bottom_countries = top_bottom_countries.merge(
    data_dict[["name", "layer", "final_unit", "direction"]],
    left_on="variable",
    right_on="name",
    how="left"
).drop(columns=["name"])

top_bottom_countries = top_bottom_countries[
    ["variable", "layer", "position", "REF_AREA", "value", "final_unit", "direction"]
]

top_bottom_countries

,variable,layer,position,REF_AREA,value,final_unit,direction
0,stem_women_share,education_pipeline,bottom_5,JPN,18.078102,percent,Higher = more women represented among tertiary...
1,stem_women_share,education_pipeline,bottom_5,CHL,20.991544,percent,Higher = more women represented among tertiary...
2,stem_women_share,education_pipeline,bottom_5,CHE,26.050484,percent,Higher = more women represented among tertiary...
3,stem_women_share,education_pipeline,bottom_5,ESP,27.498438,percent,Higher = more women represented among tertiary...
4,stem_women_share,education_pipeline,bottom_5,KOR,27.657623,percent,Higher = more women represented among tertiary...
5,stem_women_share,education_pipeline,top_5,ISL,43.291405,percent,Higher = more women represented among tertiary...
6,stem_women_share,education_pipeline,top_5,NZL,43.052057,percent,Higher = more women represented among tertiary...
7,stem_women_share,education_pipeline,top_5,GRC,41.682659,percent,Higher = more women represented among tertiary...
8,stem_women_share,education_pipeline,top_5,EST,41.545704,percent,Higher = more women represented among tertiary...
9,stem_women_share,education_pipeline,top_5,POL,40.232751,percent,Higher = more women represented among tertiary...


In [58]:
top_bottom_countries.to_csv(
    DATA_PATH + "top_bottom_countries_oecd.csv",
    index=False
)

In [59]:
top_bottom_summary_rows = []

for col in indicator_cols:
    sub = top_bottom_countries[top_bottom_countries["variable"] == col].copy()
    
    bottom = sub[sub["position"] == "bottom_5"].copy()
    top = sub[sub["position"] == "top_5"].copy()
    
    bottom_list = "; ".join(
        f"{row.REF_AREA} ({row.value:.2f})"
        for _, row in bottom.iterrows()
    )
    
    top_list = "; ".join(
        f"{row.REF_AREA} ({row.value:.2f})"
        for _, row in top.iterrows()
    )
    
    unit = sub["final_unit"].iloc[0]
    direction = sub["direction"].iloc[0]
    
    top_bottom_summary_rows.append({
        "variable": col,
        "bottom_5": bottom_list,
        "top_5": top_list,
        "unit": unit,
        "direction": direction,
    })

top_bottom_summary = pd.DataFrame(top_bottom_summary_rows)
top_bottom_summary

,variable,bottom_5,top_5,unit,direction
0,stem_women_share,JPN (18.08); CHL (20.99); CHE (26.05); ESP (27...,ISL (43.29); NZL (43.05); GRC (41.68); EST (41...,percent,Higher = more women represented among tertiary...
1,coding_gap_youth_M_minus_F,GRC (-0.37); TUR (3.71); CHL (3.92); MEX (3.93...,IRL (28.85); GBR (19.03); PRT (18.83); ESP (18...,percentage points,Higher = larger male advantage in youth coding...
2,labour_force_participation_gap_M_minus_F,ISR (4.10); LTU (4.54); FIN (4.59); SWE (4.74)...,TUR (35.63); MEX (29.69); CRI (24.53); COL (24...,percentage points,Higher = larger male advantage in labour-force...
3,gender_wage_gap_median,LUX (-8.68); COL (0.59); BEL (0.91); DNK (3.34...,KOR (28.95); ISR (20.85); JPN (20.65); EST (19...,percent,Higher = larger wage inequality / worse pay ou...
4,part_time_employment_gap_F_minus_M,ISR (-10.77); SVK (2.10); HUN (2.70); LVA (2.9...,NLD (33.26); CHE (29.03); AUT (25.86); DEU (25...,percentage points,Higher = women are more concentrated in part-t...
5,women_senior_middle_mgmt_share,JPN (13.27); LUX (16.98); ITA (24.82); DEU (26...,USA (44.33); SWE (43.80); POL (41.76); LVA (41...,percent,Higher = more women represented in senior and ...


In [60]:
# Visual

plot_df = top_bottom_countries.copy()
plot_df["country_label"] = plot_df["REF_AREA"]
variables = plot_df["variable"].unique()

fig = go.Figure()
buttons = []

for i, var in enumerate(variables):
    df_var = plot_df[plot_df["variable"] == var]
    colors = df_var["position"].map(
        {"top_5": "#0072B2", "bottom_5": "#E69F00"}
    )

    fig.add_trace(go.Bar(
        x=df_var["value"],
        y=df_var["country_label"],
        orientation="h",
        name=var,
        visible=(i == 0),
        marker_color=colors,
    ))

for i, var in enumerate(variables):
    visible = [False] * len(variables)
    visible[i] = True
    buttons.append(dict(
        label=var,
        method="update",
        args=[
            {"visible": visible},
            {"title": f"Top and bottom OECD countries by indicator: {var}"},
        ],
    ))

fig.update_layout(
    updatemenus=[
        dict(
            buttons=buttons,
            direction="down",
            x=0.0,
            xanchor="left",
            y=1.1,
            yanchor="top",
        )
    ],
    xaxis_title="Value",
    yaxis_title="Country",
    height=700,
)

fig.show()


## <a id="eda-summary-and-implications-for-hypothesis-testing"></a>EDA summary and implications for hypothesis testing

This exploratory data analysis examined the latest-available OECD-member dataset for the project "The Gender Conversion Gap: From STEM and Digital Participation to Labour-Market Power."

The term "conversion gap" is used as a project framing device, not as a literal cohort-level transition claim. The dataset does not follow the same women from STEM graduation into later management or pay outcomes. Instead, it compares recent country-level indicators to explore whether countries with more favourable STEM and digital gender-equality indicators also tend to show more favourable labour-market and leadership outcomes.

The main analysis sample contains 38 OECD member countries and excludes aggregate areas such as OECD, EU27, G7, and EU28. A broader non-aggregate sample is kept only for sensitivity checks.

Indicator coverage is sufficient for the planned analysis, but some variables have partial missingness. `stem_women_share`, `labour_force_participation_gap_M_minus_F`, and `gender_wage_gap_median` are available for all 38 OECD members. `women_senior_middle_mgmt_share` is the most limited indicator, with 30 OECD observations, followed by `coding_gap_youth_M_minus_F`, with 32 observations. Hypothesis testing should therefore use pairwise complete observations for each variable pair.

The dataset uses latest-available values rather than a synchronized same-year cross-section. The median latest-year span in the OECD-member sample is 2 years, and Estonia also has a 2-year span. Pairwise year diagnostics show that H1 has a maximum year gap of 1 year, H2 has a maximum gap of 2 years, and H4 has a maximum gap of 3 years. H3 requires the most caution because the youth coding gap and labour-force participation gap can differ by up to 8 years for Chile and 6 years for the United Kingdom.

Descriptive statistics show that women represent roughly one third of tertiary STEM graduates and roughly one third of senior and middle management positions across the OECD-member sample. Gap indicators vary more widely, especially labour-force participation gaps, part-time employment gaps, and the median gender wage gap indicator. Extreme values are treated as meaningful country-level observations unless a specific data-processing or definition issue is identified.

Estonia shows a mixed benchmark profile. It is well above the OECD median in women's share among tertiary STEM graduates and has below-median gaps in youth coding, labour-force participation, and part-time employment concentration. However, Estonia's median gender wage gap indicator is substantially above the OECD median, and women's representation in senior and middle management is below the OECD median.

Exploratory relationship plots do not suggest a simple one-dimensional alignment between STEM/digital participation indicators and labour-market power indicators. Estonia is high on women's STEM graduate share but not high on women's senior/middle management representation, and it combines high STEM representation with a high median gender wage gap indicator.

The dataset is suitable for formal hypothesis testing in the next notebook. Because the main variables are continuous country-level indicators, the main inferential methods will be correlation analysis and simple linear regression, with Spearman correlation used as a robustness check. H3 will be interpreted as exploratory because it compares the youth coding indicator with overall working-age labour-force participation.

All results should be interpreted as cross-country associations, not causal effects.
